# Sina Shamsian week 3 project: AR making landfall on Northern California

# Plotting Plan: Atmospheric River Landfall — Northern California January 2nd-10th

## 1. Objective
Produce a map showing an atmospheric river (AR) making landfall over Northern California, highlighting the moisture plume, its orientation/strength, and the landfall location/timing.

## 2. Core Variable: Integrated Vapor Transport (IVT)
IVT is the standard field for visualizing ARs.

- **Formula:** IVT = sqrt(IVTu² + IVTv²), where IVTu/IVTv are vertically integrated moisture flux components (surface to ~300 hPa).
- **AR threshold:** commonly IVT ≥ 250 kg/m/s (or 500+ kg/m/s for strong events) sustained along a plume shape.

## 3. Map Design

### Base map
- **Projection:** `cartopy.crs.PlateCarree()` for simplicity, or `cartopy.crs.LambertConformal()` centered near ~40°N, -122°W for a more "regional forecast" look.
- **Extent:** roughly 20–55°N, 160–110°W to capture the full AR plume from the Pacific source region to the CA coast. Optionally an inset/zoom panel focused on Northern California (35–43°N, 115–125°W).
- **Coastlines/borders:** `cartopy.feature.COASTLINE`, `STATES`, and your existing county shapefiles (reuse the ones from your San Diego County precip work) for the Northern CA landfall zone.

### IVT plume
- **Filled contour (`contourf`) or `pcolormesh`** of IVT magnitude, colormap suggestion: `cmocean.cm.rain` or `plt.cm.YlGnBu`/`nipy_spectral`, scale 0–1000+ kg/m/s.
- **Contour line at 250 kg/m/s** (dashed, black or white) to outline the AR boundary objectively — ties in nicely with your MODE object-detection background.

### Wind/moisture transport vectors
- Overlay `quiver` or `barbs` of IVTu/IVTv (thin/subsampled) to show transport direction — this makes the "landfall" visually obvious as arrows plunge into the CA coast.

### Landfall marker
- Mark the coastal landfall point/segment (e.g., near Point Reyes / Cape Mendocino / wherever your case study lands) with a star or highlighted coastline segment.
- Optional: annotate timestamp (valid time) and AR strength category (e.g., using Ralph et al. AR scale 1–5) as a text box.

## 4. Tools/Libraries
- `xarray` — load ERA5/West-WRF NetCDF, compute IVT if not already present
- `numpy` — vector magnitude calculations
- `matplotlib` + `cartopy` — mapping, contours, quiver
- `geopandas` — Northern CA county shapefiles (reuse pattern from NOHRSC precip plot)
- Optional: `cmocean` for AR-appropriate colormaps

## 5. Step-by-Step Workflow
1. Load ERA5/West-WRF dataset for the AR case date/time window.
2. Compute or extract IVT magnitude and components (IVTu, IVTv).
3. Subset spatially to Pacific-to-California domain.
4. Set up cartopy figure with basemap, coastlines, county shapefile overlay.
5. Plot filled IVT contours + 250 kg/m/s AR boundary contour.
6. Overlay subsampled quiver/barbs for transport direction.
7. Add landfall marker + annotation (valid time, AR category if computed).

## 6. Notes
- Will be added throughout the process of coding 


# Import important python packages

In [ ]:
import pandas as pd 
import numpy as np
import xarray as xr 
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import glob
import os
import re
from matplotlib.lines import Line2D
import cartopy.crs as ccrs
import cartopy.feature as cfeature

## Data path of the first event January 4th 2017
- We are using the 4th of January in this case because the AR only made landfall on the Northern California coast on the 4th of Janaury but was still developing out in the ocean on the 2nd and the 3rd 

In [ ]:
datapath = "/data/projects/WWRF-NRT/30YEAR-REFORECAST/MODE_verification/Raw_output/2017/2017010400/500/mode_WestWRF_240000L_20170104_000000V_000000A_obj.nc"
ds = xr.open_dataset(datapath)

# Plotting an example of the AR 
- this is just a quick plot to guage what the AR looks like on the first AR event on January 3rd, 2017

In [ ]:
plt.figure(figsize=(10, 7))
ds["fcst_raw"].plot(x="lon", y="lat", cmap="plasma_r", cbar_kwargs = {"label": "IVT (kg m$^{-1}$ s$^{-1}$)"})
plt.title("Forecast Raw Field January 4, 2017 00Z +24h")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.show()

plt.figure(figsize=(10, 7))
ds["obs_raw"].plot(x="lon", y="lat", cmap="plasma_r", cbar_kwargs = {"label": "IVT (kg m$^{-1}$ s$^{-1}$)"})
plt.title("Observed Raw Field January 4, 2017 00Z +24h")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.show()

## Full plot with Cartopy map of the landfalling AR 
- In this map I display the landfalling AR using the forecasted raw values on January 4th, 2017. I use a white dotted contour line to outline values greater than 250 kgm/s threshold 

In [ ]:
# Cartopy map of the forecast IVT field over the Pacific/Northern California domain
lon = ds["lon"].values
lat = ds["lat"].values
ivt = ds["fcst_raw"].values

fig = plt.figure(figsize=(12, 8))
ax = plt.axes(projection=ccrs.PlateCarree())
ax.set_extent([-135, -115, 30, 50], crs=ccrs.PlateCarree())

ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
ax.add_feature(cfeature.BORDERS, linewidth=0.4)
ax.add_feature(cfeature.STATES, linewidth=0.4)
ax.add_feature(cfeature.OCEAN, facecolor='#cfe3ee', zorder=0)
ax.add_feature(cfeature.LAND, facecolor='#f0efe9', zorder=0)

cf = ax.contourf(lon, lat, ivt, levels=np.arange(250, 1400, 50), cmap="plasma_r", transform=ccrs.PlateCarree())
# ax.contour(lon, lat, ivt, levels=[250], colors="white", linewidths=1.0, linestyles="--", transform=ccrs.PlateCarree())

ax.set_title("Forecast IVT of AR in 2017 \n Initial Forecast: January 4, 2017 00Z +24h \n Forecast Valid: January 5, 2017 00Z", fontsize=14)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
cbar = plt.colorbar(cf, ax=ax, pad=0.04, shrink=0.8)
cbar.set_label("IVT (kg m$^{-1}$ s$^{-1}$)")
# Red box showing the landfall region
ax = plt.gca()

lon0, lon1 = -125, -121
lat0, lat1 = 36, 41

rect = plt.Rectangle(
    (lon0, lat0),
    lon1 - lon0,
    lat1 - lat0,
    fill=False,
    edgecolor="red",
    linewidth=2,
    zorder=5
)
ax.add_patch(rect)
plt.tight_layout()
plt.show()

# AR second landfall event January 7-9th 2017

### Datapath January 7th

In [ ]:
datapath2 = "/data/projects/WWRF-NRT/30YEAR-REFORECAST/MODE_verification/Raw_output/2017/2017010700/500/mode_WestWRF_240000L_20170107_000000V_000000A_obj.nc"
ds2 = xr.open_dataset(datapath2)

### inspect dataset 

In [ ]:
ds2

## Quick plot 

In [ ]:
plt.figure(figsize=(10, 7))
ds2["fcst_raw"].plot(x="lon", y="lat", cmap="plasma_r", cbar_kwargs = {"label": "IVT (kg m$^{-1}$ s$^{-1}$)"})
plt.title("Forecast Raw Field January 7, 2017 00Z +24h")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.show()

plt.figure(figsize=(10, 7))
ds2["obs_raw"].plot(x="lon", y="lat", cmap="plasma_r", cbar_kwargs = {"label": "IVT (kg m$^{-1}$ s$^{-1}$)"})
plt.title("Observed Raw Field January 7, 2017 00Z +24h")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.show()

## Full plot with cartopy outline 

In [ ]:
# Cartopy map of the forecast IVT field over the Pacific/Northern California domain
lon2 = ds2["lon"].values
lat2 = ds2["lat"].values
ivt2 = ds2["fcst_raw"].values

fig = plt.figure(figsize=(12, 8))
ax = plt.axes(projection=ccrs.PlateCarree())
ax.set_extent([-135, -115, 30, 50], crs=ccrs.PlateCarree())

ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
ax.add_feature(cfeature.BORDERS, linewidth=0.4)
ax.add_feature(cfeature.STATES, linewidth=0.4)
ax.add_feature(cfeature.OCEAN, facecolor='#cfe3ee', zorder=0)
ax.add_feature(cfeature.LAND, facecolor='#f0efe9', zorder=0)

cf = ax.contourf(lon2, lat2, ivt2, levels=np.arange(250, 1400, 50), cmap="plasma_r", transform=ccrs.PlateCarree())
ax.contour(lon2, lat2, ivt2, levels=[250], colors="white", linewidths=1.0, linestyles="--", transform=ccrs.PlateCarree())

ax.set_title("Forecast IVT of AR in 2017 \n Initial Forecast: January 7, 2017 00Z +24h \n Forecast Valid: January 8, 2017 00Z", fontsize=14)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
cbar = plt.colorbar(cf, ax=ax, pad=0.04, shrink=0.8)
cbar.set_label("IVT (kg m$^{-1}$ s$^{-1}$)")
# Red box showing the landfall region
ax = plt.gca()

lon0, lon1 = -125, -121
lat0, lat1 = 36, 41

rect = plt.Rectangle(
    (lon0, lat0),
    lon1 - lon0,
    lat1 - lat0,
    fill=False,
    edgecolor="red",
    linewidth=2,
    zorder=5
)
ax.add_patch(rect)
plt.tight_layout()
plt.show()

## Datapath January 8th 2017

In [ ]:
datapath3 = "/data/projects/WWRF-NRT/30YEAR-REFORECAST/MODE_verification/Raw_output/2017/2017010800/500/mode_WestWRF_240000L_20170108_000000V_000000A_obj.nc"
ds3 = xr.open_dataset(datapath3)

## Quickplot 

In [ ]:
plt.figure(figsize=(10, 7))
ds3["fcst_raw"].plot(x="lon", y="lat", cmap="plasma", cbar_kwargs = {"label": "IVT (kg m$^{-1}$ s$^{-1}$)"})
plt.title("Forecast Raw Field January 8, 2017 00Z +24h")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.show()

plt.figure(figsize=(10, 7))
ds3["obs_raw"].plot(x="lon", y="lat", cmap="plasma", cbar_kwargs = {"label": "IVT (kg m$^{-1}$ s$^{-1}$)"})
plt.title("Observed Raw Field January 8, 2017 00Z +24h")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.show()

## Full plot with Cartopy outline 

In [ ]:
# Cartopy map of the forecast IVT field over the Pacific/Northern California domain
lon2 = ds3["lon"].values
lat2 = ds3["lat"].values
ivt2 = ds3["fcst_raw"].values

fig = plt.figure(figsize=(12, 8))
ax = plt.axes(projection=ccrs.PlateCarree())
ax.set_extent([-135, -115, 30, 50], crs=ccrs.PlateCarree())

ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
ax.add_feature(cfeature.BORDERS, linewidth=0.4)
ax.add_feature(cfeature.STATES, linewidth=0.4)
ax.add_feature(cfeature.OCEAN, facecolor='#cfe3ee', zorder=0)
ax.add_feature(cfeature.LAND, facecolor='#f0efe9', zorder=0)

cf = ax.contourf(lon2, lat2, ivt2, levels=np.arange(0, 1001, 50), cmap="plasma", transform=ccrs.PlateCarree())
ax.contour(lon2, lat2, ivt2, levels=[250], colors="white", linewidths=1.0, linestyles="--", transform=ccrs.PlateCarree())

ax.set_title("Forecast IVT and AR boundary\nJanuary 8, 2017 00Z +24h")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
cbar = plt.colorbar(cf, ax=ax, pad=0.04, shrink=0.8)
cbar.set_label("IVT (kg m$^{-1}$ s$^{-1}$)")
# Red box showing the landfall region
ax = plt.gca()

lon0, lon1 = -125, -121
lat0, lat1 = 36, 41

rect = plt.Rectangle(
    (lon0, lat0),
    lon1 - lon0,
    lat1 - lat0,
    fill=False,
    edgecolor="red",
    linewidth=2,
    zorder=5
)
ax.add_patch(rect)
plt.tight_layout()
plt.show()